In [ ]:
# the building blocks
prompt = ChatPromptTemplate.from_template("""Answer the question based on the context below.
you must give to the user the listing in the documents context as recommendations. The user is looking 
for listings and make the reservation of the listing.

Context: {context}
                                            
Question: {question}

Answer:
""")

# Initialize the OpenAI embedding model
embeddings = OpenAIEmbeddings()

#Create a ChromDB vector database
vector_store = Chroma(
    collection_name='smartbnb_vector_store',
    embedding_function=embeddings,
    persist_directory='/Users/gblasd/Documents/SmartBnB/db/chroma_db'
)

print(vector_store.__class__)


# create retriver
retriever = vector_store.as_retriever()

# Initialize the model
llm_model = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

# metadata description
description = "Brief summary of listing"

# metadata schema
fields = [
    AttributeInfo(
        name="price",
        description="price per night of the listing",
        type="integer"
    ),
    AttributeInfo(
        name="property_type",
        description="Type of property",
        type="string"
    ),
    AttributeInfo(
        name="room_type",
        description="Type of room",
        type="string"
    ), 
    AttributeInfo(
        name="neighbourhood_cleansed",
        description="Tneighbourhood where the listing is",
        type="string"
    ), 
    AttributeInfo(
        name="review_scores_accuracy",
        description="A 1-5 rating for the listing",
        type="float"
    ),
]

retriever = SelfQueryRetriever.from_llm(
    llm=llm_model,
    vectorstore=vector_store,
    metadata_field_info=fields,
    document_contents=description
)

# Query transform
rewrite_prompt = ChatPromptTemplate.from_template("""Provide a better search
query for web search engine to answer the given question, end the queries
with `**`. Question: {x} Answer:""")
def parse_rewriter_output(message):
    return message.content.strip('"').strip("**")
# Rewrite-Retrieve-Read
# LCEL Declarative conposition, optimized execution plan, 
# we dont need to use invoke/stream/batch, it's automatic 
rewriter = rewrite_prompt | llm_model | parse_rewriter_output 


# combine them in a function 
# @chain decorator adds the same Runnable interface for any function you write
# Imperative composition, code into functions and classes
@chain
def chatbot(input):
    # rewrite the query, only for the retriever nd get the relevant documents
    new_query = rewriter.invoke(input)

    # fetch relevant documents
    # docs = retriever.get_relevant_documents(input)
    docs = retriever.invoke(input)

    # format prompt
    formatted = prompt.invoke({"context":docs, "question":input})

    # generate answer
    answer = llm_model.invoke(formatted)

    return {"answer": answer, "docs": docs}
    #return answer


# use it
chatbot.invoke("Can you show me 5 listings near from the university with roof garden, price between 1500 and 3000, with location neighbourhood in Tlalpan")

<class 'langchain_chroma.vectorstores.Chroma'>


InvalidArgumentError: Collection expecting embedding with dimension of 384, got 1536